# ETL Silver to Gold

Organizando e Populando Star Schema no PostgreSQL a partir dos dados limpos do Silver Layer.

---

# 1. Bibliotecas

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

import psycopg2
from psycopg2.extras import execute_batch

# 2. Conectando no Banco

In [2]:
spark = SparkSession.builder \
    .appName("gold") \
    .config(
        "spark.jars.packages",
        "org.postgresql:postgresql:42.7.3"
    ) \
    .getOrCreate()


jdbc_url = "jdbc:postgresql://localhost:5432/cars_trips"

properties = {
    "user": "postgres",
    "password": "postgres",
    "driver": "org.postgresql.Driver"
}

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/02/03 20:39:08 WARN Utils: Your hostname, CyberCore, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/02/03 20:39:08 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/danielsousa/car_rides_analytics/venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/danielsousa/.ivy2.5.2/cache
The jars for the packages stored in: /home/danielsousa/.ivy2.5.2/jars
org.postgresql#postgresql added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-7923477f-3a70-4e9a-a641-851db21fa857;1.0
	confs: [default]
	found org.postgresql#postgresql;42.7.3 in central
	found org.checkerframework#checker-qual;3.42.0 in central
:: resolution report :: resolve 153ms :: artifacts dl 8ms
	:: modules in use:
	org.checke

# 3. Leitura da Silver

In [3]:
df_silver = spark.read.jdbc(
    url=jdbc_url,
    table="silver.road_trips",
    properties=properties
)

df_silver = df_silver.withColumn(
    "time",
    F.date_format(F.col("time"), "HH:mm:ss")
)

df_silver.show()

+---+----------+--------+--------------------+-----------+------------+-----------------+-------------+------------------+-----------------------+--------------------+--------------------+----------------+-----------------------+-------------+--------------+-------------+--------------+---------------+
| id|      date|    time|      booking_status|customer_id|vehicle_type|  pickup_location|drop_location|driver_time_accept|passenger_time_boarding|canceled_by_customer|  canceled_by_driver|incomplete_rides|incomplete_rides_reason|booking_value|payment_method|ride_distance|driver_ratings|customer_rating|
+---+----------+--------+--------------------+-----------+------------+-----------------+-------------+------------------+-----------------------+--------------------+--------------------+----------------+-----------------------+-------------+--------------+-------------+--------------+---------------+
|  1|2024-07-26|14:00:00|  Canceled by Driver|  CID713523| Prime Sedan|      Tumkur Road

# 4. Dimensão de Rotas (DIM_ROT)

In [4]:
print("Populando DIM_ROT")

dim_rot_data = df_silver.select(
    F.col("pickup_location").alias("pck_loc"),
    F.col("drop_location").alias("drp_loc")
).distinct()

dim_rot_data.show()

dim_rot_data.write.jdbc(
    url=jdbc_url,
    table="dw.dim_rot",
    mode="append",
    properties=properties
)

print(f"✓ {dim_rot_data.count()} rotas inseridas")

Populando DIM_ROT
+---------------+--------------------+
|        pck_loc|             drp_loc|
+---------------+--------------------+
|      Yelahanka|         Rajajinagar|
|     Hosur Road|         Koramangala|
|   Shivajinagar|     Electronic City|
|    Shantinagar|         Magadi Road|
|       Majestic|              Hennur|
|Padmanabhanagar|         Chamarajpet|
|  Richmond Town|            RT Nagar|
|         Hennur|             Varthur|
|   Yeshwanthpur|          Hosur Road|
|  Langford Town|        Shivajinagar|
|        Kengeri|            Cox Town|
|     Hosur Road|Rajarajeshwari Nagar|
|      Jayanagar|         Vijayanagar|
|        Varthur|          Whitefield|
|         Ulsoor|            JP Nagar|
|  Richmond Town|Rajarajeshwari Nagar|
|    Mysore Road|        Marathahalli|
|  Sarjapur Road|              Hebbal|
|  Sahakar Nagar|     Padmanabhanagar|
|         Hennur|            RT Nagar|
+---------------+--------------------+
only showing top 20 rows
✓ 2500 rotas inserida

# 5. Dimensão de Customer (DIM_CST)

In [5]:
print("Populando DIM_CST")

# Pega o tempo para poder fazer o filtro
df_with_time = df_silver.withColumn(
    "date_time_safe",
    F.to_timestamp(F.concat_ws(" ", F.col("date"), F.col("time")))
)

window = Window.partitionBy("customer_id").orderBy(F.col("date_time_safe").desc())

dim_cst_data = (
    df_with_time
    .withColumn("rn", F.row_number().over(window))
    .filter("rn = 1")
    .select(
        F.col("customer_id").alias("cst_idf"),
        F.col("customer_rating").alias("cst_rtg")
    )
    .dropDuplicates(["cst_idf"])
)

dim_cst_data.show()

dim_cst_data.write.jdbc(
    url=jdbc_url,
    table="dw.dim_cst",
    mode="append",
    properties=properties
)

print(f"✓ {dim_cst_data.count()} clientes únicos inseridos")

Populando DIM_CST


+---------+-------+
|  cst_idf|cst_rtg|
+---------+-------+
|CID100033|   4.20|
|CID100034|   NULL|
|CID100043|   3.20|
|CID100046|   3.50|
|CID100052|   NULL|
|CID100070|   3.40|
|CID100084|   3.60|
|CID100085|   3.20|
|CID100094|   4.80|
|CID100102|   NULL|
|CID100118|   3.80|
|CID100128|   4.40|
|CID100133|   NULL|
|CID100140|   4.10|
|CID100146|   NULL|
|CID100156|   NULL|
|CID100160|   NULL|
|CID100169|   NULL|
|CID100206|   3.30|
|CID100212|   NULL|
+---------+-------+
only showing top 20 rows


✓ 85727 clientes únicos inseridos


# 6. Dimensão de Status de Viagem (DIM_STS)

In [6]:
print("Populando DIM_STS")

dim_sts_data = df_silver.select(
    F.col("booking_status").alias("bkg_sts"),
    F.col("canceled_by_customer").alias("cld_cst"),
    F.col("canceled_by_driver").alias("cld_drv"),
    F.col("incomplete_rides").alias("icm_rid"),
    F.col("incomplete_rides_reason").alias("icm_rid_rsn")
).distinct()

dim_sts_data.show()

dim_sts_data.write.jdbc(
    url=jdbc_url,
    table="dw.dim_sts",
    mode="append",
    properties=properties
)

print(f"✓ {dim_sts_data.count()} combinações de status inseridas")

Populando DIM_STS
+--------------------+--------------------+--------------------+-------+-----------------+
|             bkg_sts|             cld_cst|             cld_drv|icm_rid|      icm_rid_rsn|
+--------------------+--------------------+--------------------+-------+-----------------+
|             Success|                NULL|                NULL|   true|  Customer Demand|
|  Canceled by Driver|                NULL|Customer was coug...|   NULL|             NULL|
|  Canceled by Driver|                NULL|More than permitt...|   NULL|             NULL|
|Canceled by Customer|Driver asked to c...|                NULL|   NULL|             NULL|
|Canceled by Customer|     Change of plans|                NULL|   NULL|             NULL|
|             Success|                NULL|                NULL|   true|      Other Issue|
|Canceled by Customer|Driver is not mov...|                NULL|   NULL|             NULL|
|  Canceled by Driver|                NULL|Customer related ...|   NULL|

# 7. Dimensão da Data e Hora da Viagem (DIM_TIM)

Usando o `psycopg2` por conta de erros com time no spark

In [7]:
print("Populando DIM_TIM")

dim_tim_data = df_silver.select(
    F.date_format("date", "yyyy-MM-dd").cast("date").alias("dat"),
    F.date_format(F.col("time"), "HH:mm:ss").alias("tim")
).distinct()

dim_tim_data.show()

rows = [(r["dat"], r["tim"]) for r in dim_tim_data.collect()]

conn = psycopg2.connect(
    host="localhost",
    database="cars_trips",
    user="postgres",
    password="postgres"
)

cursor = conn.cursor()

sql = """
    INSERT INTO dw.dim_tim (dat, tim)
    VALUES (%s, %s)
    ON CONFLICT DO NOTHING;
"""

execute_batch(cursor, sql, rows, page_size=5000)

conn.commit()
cursor.close()
conn.close()

print(f"Inseridos {len(rows)} registros em dw.dim_tim via psycopg2.")

Populando DIM_TIM


+----------+--------+
|       dat|     tim|
+----------+--------+
|2024-07-24|00:10:00|
|2024-07-11|02:28:00|
|2024-07-18|07:52:00|
|2024-07-15|23:20:00|
|2024-07-22|14:23:00|
|2024-07-11|18:33:00|
|2024-07-05|10:06:00|
|2024-07-26|01:07:00|
|2024-07-16|23:00:00|
|2024-07-04|13:54:00|
|2024-07-25|15:19:00|
|2024-07-13|05:17:00|
|2024-07-23|19:57:00|
|2024-07-09|11:54:00|
|2024-07-27|22:02:00|
|2024-07-26|20:13:00|
|2024-07-26|19:53:00|
|2024-07-22|20:31:00|
|2024-07-02|02:07:00|
|2024-07-30|09:43:00|
+----------+--------+
only showing top 20 rows
Inseridos 39051 registros em dw.dim_tim via psycopg2.


# 8. Dimensão de Pagamento (DIM_PAY)

In [8]:
print("Populando DIM_PAY")

dim_pay_data = df_silver.select(
    F.col("payment_method").alias("pay_mtd")
).distinct()

dim_pay_data.show()

dim_pay_data.write.jdbc(
    url=jdbc_url,
    table="dw.dim_pay",
    mode="append",
    properties=properties
)

print(f"✓ {dim_pay_data.count()} métodos de pagamento inseridos")

Populando DIM_PAY
+-----------+
|    pay_mtd|
+-----------+
|Credit Card|
|       Cash|
| Debit Card|
|        UPI|
|       NULL|
+-----------+

✓ 5 métodos de pagamento inseridos


# 9. Dimensão de Veículos (DIM_VEC)

In [9]:
print("Populando DIM_VEC")

dim_vec_data = df_silver.select(
    F.col("vehicle_type").alias("vec_typ")
).distinct()

dim_vec_data.show()

dim_vec_data.write.jdbc(
    url=jdbc_url,
    table="dw.dim_vec",
    mode="append",
    properties=properties
)

print(f"✓ {dim_vec_data.count()} tipos de veículo inseridos")

Populando DIM_VEC
+-----------+
|    vec_typ|
+-----------+
|       Bike|
|       Mini|
| Prime Plus|
|       Auto|
|Prime Sedan|
|      eBike|
|  Prime SUV|
+-----------+

✓ 7 tipos de veículo inseridos


# 10. Carregar Dimesões e Inserir na Fato

In [10]:
print("Populando tabela fato FAC_TRP com LEFT JOINS")

dim_rot_db = spark.read.jdbc(url=jdbc_url, table="dw.dim_rot", properties=properties)
dim_cst_db = spark.read.jdbc(url=jdbc_url, table="dw.dim_cst", properties=properties)
dim_sts_db = spark.read.jdbc(url=jdbc_url, table="dw.dim_sts", properties=properties)
dim_tim_db = spark.read.jdbc(url=jdbc_url, table="dw.dim_tim", properties=properties)
dim_pay_db = spark.read.jdbc(url=jdbc_url, table="dw.dim_pay", properties=properties)
dim_vec_db = spark.read.jdbc(url=jdbc_url, table="dw.dim_vec", properties=properties)


dim_tim_db = dim_tim_db.withColumn(
    "tim_string", 
    F.date_format(F.col("tim"), "HH:mm:ss")
)

df_fact = df_silver \
    .withColumn("time_string", F.date_format(F.col("time"), "HH:mm:ss")) \
    .withColumn("cld_cst_clean", F.coalesce(F.col("canceled_by_customer"), F.lit("__NULL__"))) \
    .withColumn("cld_drv_clean", F.coalesce(F.col("canceled_by_driver"), F.lit("__NULL__"))) \
    .withColumn("icm_rid_rsn_clean", F.coalesce(F.col("incomplete_rides_reason"), F.lit("__NULL__")))

dim_sts_db = dim_sts_db \
    .withColumn("cld_cst_clean", F.coalesce(F.col("cld_cst"), F.lit("__NULL__"))) \
    .withColumn("cld_drv_clean", F.coalesce(F.col("cld_drv"), F.lit("__NULL__"))) \
    .withColumn("icm_rid_rsn_clean", F.coalesce(F.col("icm_rid_rsn"), F.lit("__NULL__")))

df_with_keys = (
    df_fact
    .join(dim_rot_db,
          (df_fact.pickup_location == dim_rot_db.pck_loc) &
          (df_fact.drop_location == dim_rot_db.drp_loc), "left")
    .join(dim_cst_db,
          df_fact.customer_id == dim_cst_db.cst_idf, "left")
    .join(dim_sts_db,
          (df_fact.booking_status == dim_sts_db.bkg_sts) &
          (df_fact.cld_cst_clean == dim_sts_db.cld_cst_clean) &
          (df_fact.cld_drv_clean == dim_sts_db.cld_drv_clean) &
          (df_fact.incomplete_rides == dim_sts_db.icm_rid) &
          (df_fact.icm_rid_rsn_clean == dim_sts_db.icm_rid_rsn_clean), "left")
    .join(dim_tim_db,
          (df_fact.date == dim_tim_db.dat) &
          (df_fact.time_string == dim_tim_db.tim_string), "left")
    .join(dim_pay_db,
          df_fact.payment_method == dim_pay_db.pay_mtd, "left")
    .join(dim_vec_db,
          df_fact.vehicle_type == dim_vec_db.vec_typ, "left")
)

fact_data = df_with_keys.select(
    F.col("sts_key").alias("sts_srk"),
    F.col("tim_key").alias("tim_srk"),
    F.col("cst_key").alias("cst_srk"),
    F.col("pay_key").alias("pay_srk"),
    F.col("vec_key").alias("vec_srk"),
    F.col("rot_key").alias("rot_srk"),
    F.col("ride_distance").alias("rid_dst"),
    F.col("driver_ratings").alias("drv_rtg"),
    F.col("booking_value").alias("bkg_val"),
    F.col("driver_time_accept").alias("drv_tim_acp"),
    F.col("passenger_time_boarding").alias("psg_tim_bdg")
)

fact_count = fact_data.count()
print(f"✓ {fact_count:,} viagens prontas para inserir")

fact_data.write.jdbc(
    url=jdbc_url,
    table="dw.fac_trp",
    mode="append",
    properties=properties
)

print(f"✓ {fact_count:,} viagens inseridas na tabela fato")

Populando tabela fato FAC_TRP com LEFT JOINS
✓ 92,900 viagens prontas para inserir


✓ 92,900 viagens inseridas na tabela fato
